# Big Data Analytics
Praktikum Sommersemester 2023. <small>Version 1.1</small>

**Aufgabe 2: Abfragen mit Apache Drill** 

Machen Sie sich mit Apache Drill vertraut. Lösen Sie die Aufgaben in `sqlline` oder in einem Jupyter Notebook. Nutzen Sie die markierten Zellen im vorliegenden Notebook `BDA1_A2_Drill` für die Lösung und laden Sie es in Ilias hoch.

----

## Vorbereitung
* Erzeugen Sie den Ordner `work/drill-driver` im JupyterLab Workspace
* Legen Sie eine neue Datei `odbc.ini` dort an. Sie finden ein Beispiel unter `Big_Data_Analytics_1/public/drill-driver/`


In Drill sind mehrere Datenmengen konfiguriert und für Sie verwendbar:

* **dfs.data.\`co2data.tsv\`**<br>DFS Datenquelle in Form eines Datensatz an Sensordaten
* **dfs.bdapi.labels**<br>HTTP Datenquelle mit Sensornamen und -Positionen
* **dfs.bdapi.sensors**<br>HTTP Datenquelle mit einer fixierten Liste von JSON Objekten mit Sensordaten.
* **dfs.bdapi.sensorslastday**<br>HTTP Datenqelle mit einem JSON Objekt, das dynamische Sensordaten anbietet.
* **dfs.weather.sunrise**<br>HTTP Storage einer API, die Zeiten des Sonnenauf- und Sonnenuntergangs an einer gegebenen Position (fields lat und lon) zu einem Datum (field date) zurückgibt. Mehr Infos dazu auf der Webseite zur [sunrise-sunset.org/api](https://sunrise-sunset.org/api).

## Aufgabe 2 a

Verschaffen Sie sich einen Überblick über die Sensordaten, die Sie im Drill unter ``dfs.data.`co2data.tsv` `` finden. Beantworten Sie die folgenden
Fragen, indem Sie jeweils eine SQL-Query gegen das Drill Cluster ausführen:
    
1. Wieviele verschiedene Sensoren (angegeben im Feld _source_) enhält die Datenmenge?
2. Wieviele Datenpunkte je Sensor liegen vor? Geben Sie sowohl Sensor als auch Anzahl aus.
3. Bereiten Sie in einer SQL-Query die Werte in einzelnen Spalten so vor, dass sie sinnvolle Datentypen aufweisen:
    1. Sowohl _humidity_, _temperature_ als auch _co2_ sollen als auf zwei Nachkommastellen gerundete Fließkommazahlen verfügbar sein.
    2. Die erste Spalte gibt den Zeitstempel als Unix Epoch mit Mikrosekunden an. Machen Sie daraus einen Drill Timestamp
4. Was ist der höchste, und was der niedrigste Temperaturwert? Beide Werte sollen zusammen in einer Query angefragt werden. Kennzeichnen Sie die beiden Felder mit einem sprechenden Namen.
5. Was ist der durchschnittliche CO<sub>2</sub>-Wert je Sensor in der Datenmenge?


In [6]:
# Aufgabe 2a: 1.
import pyodbc
from tabulate import tabulate

def connect(dsn='drill'):
    """opens the connection to given DSN"""
    conn = pyodbc.connect("DSN="+dsn, autocommit=True)
    if not conn.closed:
        print(f"connected to drillbit {dsn}")
        return conn
    else:
        print(f"could not connect to {dsn}")
        return None

    
con = connect()

csr = con.cursor()
for row in csr.execute("SELECT count(distinct columns[1]) AS sensor FROM dfs.data.`co2data.tsv` where columns[1] <> 'serial_number'"):
    print(f"Die Datenmenge enthält {row.sensor} verschiedene Sensoren.")

connected to drillbit drill
Die Datenmenge enthält 22 verschiedene Sensoren.


In [10]:
# Aufgabe 2a: 2.
csr.execute("SELECT columns[1] AS sensor, COUNT(*) AS Datenpunkte FROM dfs.data.`co2data.tsv` where columns[1] <> 'serial_number' GROUP BY columns[1]")

results = csr.fetchall()

data = [(row.sensor,row.Datenpunkte) for row in results]

table_headers = ["Sensor", "Datenpunkte"]
print(tabulate(data, headers=table_headers, tablefmt="grid"))

+---------------------------+---------------+
| Sensor                    |   Datenpunkte |
+===========================+===============+
| s_8caab57a6dd9            |            11 |
+---------------------------+---------------+
| s_10521c0202ab_284839     |          2064 |
+---------------------------+---------------+
| s_e8db84c5f33d            |            83 |
+---------------------------+---------------+
| s_e8db84c5f771_           |             2 |
+---------------------------+---------------+
| s_10521c01cf19_262520     |        385103 |
+---------------------------+---------------+
| s_d8bfc0147061_           |             1 |
+---------------------------+---------------+
| s_8caab57c3e19_           |             1 |
+---------------------------+---------------+
| s_d8bfc014724e_262793     |       2103522 |
+---------------------------+---------------+
| s_8caab57c3e19_282028     |       1561045 |
+---------------------------+---------------+
| s_e8db84c5f33d_281913     |     

In [34]:
# Aufgabe 2a: 3.
for row in csr.execute("""SELECT query.drill_timestamp as Zeit, ROUND(query.co2, 2) AS CO2, ROUND(query.temperature, 2) AS Temperatur, ROUND(query.humidity, 2) AS Humidity FROM (
SELECT CAST(FROM_UNIXTIME(columns[6] / 1) as TIMESTAMP) AS drill_timestamp, CAST(COLUMNS[3] AS FLOAT) as co2,
CAST(COLUMNS[4] AS FLOAT) as temperature,
CAST(COLUMNS[5] AS FLOAT) as humidity
FROM dfs.data.`co2data.tsv`) as query offset 1
"""):
    print(f"Zeit: {row.Zeit} | CO2: {row.CO2} | Temperatur: {row.Temperatur} | Humidity: {row.Humidity}")

Zeit: 2021-04-01 10:08:48 | CO2: 420.0 | Temperatur: 23.0 | Humidity: 36.0
Zeit: 2021-04-01 10:08:56 | CO2: 421.0 | Temperatur: 24.0 | Humidity: 32.0
Zeit: 2021-04-01 10:08:59 | CO2: 420.0 | Temperatur: 24.0 | Humidity: 32.0
Zeit: 2021-04-01 10:09:21 | CO2: 651.0 | Temperatur: 20.0 | Humidity: 44.0
Zeit: 2021-04-01 10:09:29 | CO2: 422.0 | Temperatur: 23.0 | Humidity: 36.0
Zeit: 2021-04-01 10:09:32 | CO2: 422.0 | Temperatur: 23.0 | Humidity: 36.0
Zeit: 2021-04-01 10:09:39 | CO2: 423.0 | Temperatur: 23.0 | Humidity: 36.0
Zeit: 2021-04-01 10:09:47 | CO2: 436.0 | Temperatur: 22.0 | Humidity: 41.0
Zeit: 2021-04-01 10:10:01 | CO2: 653.0 | Temperatur: 20.0 | Humidity: 44.0
Zeit: 2021-04-01 10:10:06 | CO2: 653.0 | Temperatur: 20.0 | Humidity: 44.0
Zeit: 2021-04-01 10:10:13 | CO2: 652.0 | Temperatur: 20.0 | Humidity: 44.0
Zeit: 2021-04-01 10:10:31 | CO2: 433.0 | Temperatur: 22.0 | Humidity: 41.0
Zeit: 2021-04-01 10:10:40 | CO2: 771.0 | Temperatur: 20.0 | Humidity: 49.0
Zeit: 2021-04-01 10:10:43

In [7]:
# Aufgabe 2a: 4.
csr.execute("""select MAX(query.colmaxtemp) as MAX_TEMP, MIN(query.colmintemp) as MIN_TEMP 
                       FROM (SELECT CAST(columns[4] as FLOAT) as colmaxtemp, CAST(columns[4] as FLOAT) as colmintemp from dfs.data.`co2data.tsv` 
                       where columns[4] <> 'temperature_celsius' and columns[4] <> 'null') as query""")


results = csr.fetchall()

data = [(row.MAX_TEMP, row.MIN_TEMP) for row in results]

table_headers = ["Maximale Temperatur", "Minimale Temperatur"]
print(tabulate(data, headers=table_headers, tablefmt="grid"))

+-----------------------+-----------------------+
|   Maximale Temperatur |   Minimale Temperatur |
+=======================+=======================+
|                    36 |                    -1 |
+-----------------------+-----------------------+


In [16]:
# Aufgabe 2a: 5.
csr.execute("""SELECT columns[1] as sensor, AVG(CAST(columns[3] AS FLOAT)) as avgco2 from dfs.data.`co2data.tsv` 
                       where columns[3] NOT IN ('co2_ppm','null') AND columns[1] <> 'serial_number' group by columns[1]""")

results = csr.fetchall()

data = [(row.sensor, row.avgco2) for row in results]

table_headers = ["Sensor", "Durchschnittlicher CO2-Wert"]
print(tabulate(data, headers=table_headers, tablefmt="grid"))

+---------------------------+-------------------------------+
| Sensor                    |   Durchschnittlicher CO2-Wert |
+===========================+===============================+
| s_8caab57cc961_284337     |                       470.793 |
+---------------------------+-------------------------------+
| s_e8db84c5f33d_           |                       488.769 |
+---------------------------+-------------------------------+
| s_8caab57cc961_           |                       429.286 |
+---------------------------+-------------------------------+
| s_3c6105d3abae_           |                       832     |
+---------------------------+-------------------------------+
| s_10521c0202ab_284839     |                       436.764 |
+---------------------------+-------------------------------+
| s_e8db84c5f33d            |                      1403.23  |
+---------------------------+-------------------------------+
| s_d8bfc0147061_283903     |                       628.634 |
+-------

## Aufgabe 2 b

1. Verknüpfen Sie die Daten aus ``dfs.data.`co2data.tsv` ``  mit den Daten aus `bdapi.labels`. Geben Sie, wenn möglich, den Sensornamen ( _name_ ) und Besitzer ( _owner_ ) zu jeder Seriennummer aus. Falls Daten fehlen, geben Sie die Seriennummer dennoch aus.<br>Hinweis: Die Seriennummern liegen bei den beiden Datenmengen in unterschiedlicher Repräsentation vor. Untersuchen Sie die Datenmengen und ändern Sie Ihre Query so ab, dass die Seriennummern korrekt verknüpft werden. 
2. Überführen Sie diese Query (aus 2b) 1) in eine View. Sie haben Schreibrecht im Workspace `dfs.tmp`. Legen Sie darin eine View namens `labels_<ihre Ilias-ID>` an.<br>Beispiel: Lautet Ihre Ilias-ID `mr1337s` nutzen Sie `labels_mr1337s`. Falls diese View bereits existiert soll sie überschrieben werden!
3. Nutzen Sie Ihre neu angelegte View nun und zeigen Sie alle Sensoren des Besitzers `Elsen` an.
4. Wann ein Sensor eines unbekannten Besitzers (`unknown`) zum ersten Mal gesendet?
5. Lassen Sie sich den physischen Plan für die Query aus 2b 4. anzeigen. 

In [53]:
# Aufgabe 2b: 1.
csr.execute("""
SELECT distinct UPPER(SUBSTRING(co2data.columns[1], 3, 12)) AS Sensor, l.name as Sensorname, l.owner as Besitzer FROM dfs.data.`co2data.tsv` AS co2data 
LEFT JOIN bdapi.labels AS l ON UPPER(SUBSTRING(co2data.columns[1], 3, 12)) = l.serial where co2data.columns[1] <> 'serial_number'
""")

results = csr.fetchall()

data = [(row.Sensor, row.Sensorname, row.Besitzer) for row in results]

table_headers = ["Sensor", "Sensorname", "Besitzer"]
print(tabulate(data, headers=table_headers, tablefmt="grid"))

+--------------+--------------------+------------+
| Sensor       | Sensorname         | Besitzer   |
+==============+====================+============+
| 8CAAB57C3E19 | Main Station S2    | Elsen      |
+--------------+--------------------+------------+
| D8BFC0147061 | Commerce Center S8 | Unknown    |
+--------------+--------------------+------------+
| 8CAAB57CC961 | Main Station S3    | Elsen      |
+--------------+--------------------+------------+
| 8CAAB57A6DD9 | Admin S3           | Remmy      |
+--------------+--------------------+------------+
| E8DB84C5F33D | Commerce Center S6 | Unknown    |
+--------------+--------------------+------------+
| D8BFC014724E | Main Station S1    | Elsen      |
+--------------+--------------------+------------+
| E8DB84C5F771 | Commerce Center S7 | Unknown    |
+--------------+--------------------+------------+
| 10521C0202AB | Main Station S4    | Elsen      |
+--------------+--------------------+------------+
| 3C6105D3ABAE | Main Station S

In [54]:
# Aufgabe 2b: 2.
csr.execute("""
CREATE OR REPLACE VIEW dfs.tmp.labels_at6584s AS
SELECT distinct UPPER(SUBSTRING(co2data.columns[1], 3, 12)) AS Sensor, l.name as Sensorname, l.owner as Besitzer FROM dfs.data.`co2data.tsv` AS co2data 
LEFT JOIN bdapi.labels AS l ON UPPER(SUBSTRING(co2data.columns[1], 3, 12)) = l.serial where co2data.columns[1] <> 'serial_number'
""")

print(f"View wurde erfolgreich erstellt oder aktualisiert.")

View wurde erfolgreich erstellt oder aktualisiert.


In [61]:
# Aufgabe 2b: 3.
csr.execute("SELECT * FROM dfs.tmp.labels_at6584s where Besitzer = 'Elsen'")

results = csr.fetchall()

data = [(row.Sensor, row.Sensorname, row.Besitzer) for row in results]

table_headers = ["Sensor", "Sensorname", "Besitzer"]
print(tabulate(data, headers=table_headers, tablefmt="grid"))

+--------------+-----------------+------------+
| Sensor       | Sensorname      | Besitzer   |
+==============+=================+============+
| D8BFC014724E | Main Station S1 | Elsen      |
+--------------+-----------------+------------+
| 8CAAB57C3E19 | Main Station S2 | Elsen      |
+--------------+-----------------+------------+
| 8CAAB57CC961 | Main Station S3 | Elsen      |
+--------------+-----------------+------------+
| 3C6105D3ABAE | Main Station S5 | Elsen      |
+--------------+-----------------+------------+
| 10521C0202AB | Main Station S4 | Elsen      |
+--------------+-----------------+------------+


In [4]:
# Aufgabe 2b: 4.
csr.execute("""SELECT UPPER(SUBSTRING(co2data.columns[1], 3, 12)) AS Sensor, l.name as Sensorname, l.owner as Besitzer, 
MIN(CAST(FROM_UNIXTIME(co2data.columns[6] / 1) as TIMESTAMP)) as sendung FROM dfs.data.`co2data.tsv` AS co2data LEFT JOIN bdapi.labels AS l ON 
UPPER(SUBSTRING(co2data.columns[1], 3, 12)) = l.serial where co2data.columns[1] <> 'serial_number' AND l.owner = 'Unknown' AND co2data.columns[6] <> 'timestamp' group by UPPER(SUBSTRING(co2data.columns[1], 3, 12)), l.name, l.owner limit 1""")

results = csr.fetchall()

data = [(row.Sensor, row.Sensorname, row.Besitzer, row.sendung) for row in results]

table_headers = ["Sensor", "Sensorname", "Besitzer", "Sendungszeitpunkt"]
print(tabulate(data, headers=table_headers, tablefmt="grid"))

+--------------+--------------------+------------+---------------------+
| Sensor       | Sensorname         | Besitzer   | Sendungszeitpunkt   |
+==============+====================+============+=====================+
| E8DB84C5F33D | Commerce Center S6 | Unknown    | 2021-03-30 09:50:05 |
+--------------+--------------------+------------+---------------------+


In [45]:
# Aufgabe 2b: 5.
sql_query = """EXPLAIN PLAN FOR SELECT UPPER(SUBSTRING(co2data.columns[1], 3, 12)) AS Sensor, l.name as Sensorname, l.owner as Besitzer, 
MIN(CAST(FROM_UNIXTIME(co2data.columns[6] / 1) as TIMESTAMP)) as sendung FROM dfs.data.`co2data.tsv` AS co2data LEFT JOIN bdapi.labels AS l ON 
UPPER(SUBSTRING(co2data.columns[1], 3, 12)) = l.serial where co2data.columns[1] <> 'serial_number' AND l.owner = 'Unknown' AND co2data.columns[6] <> 'timestamp' group by UPPER(SUBSTRING(co2data.columns[1], 3, 12)), l.name, l.owner limit 1"""
csr.execute(sql_query)

# Ergebnis abrufen
result = csr.fetchall()

# Ausgabe des EXPLAIN PLAN
for row in result:
    print(row)

('00-00    Screen\n00-01      Project(Sensor=[$0], Sensorname=[$1], Besitzer=[$2], sendung=[$3])\n00-02        ComplexToJson\n00-03          Project(Sensor=[$0], Sensorname=[$1], Besitzer=[$2], sendung=[$3])\n00-04            SelectionVectorRemover\n00-05              Limit(fetch=[1])\n00-06                UnionExchange\n01-01                  SelectionVectorRemover\n01-02                    Limit(fetch=[1])\n01-03                      HashAgg(group=[{0, 1, 2}], sendung=[MIN($3)])\n01-04                        Project(Sensor=[$0], Sensorname=[$1], Besitzer=[$2], sendung=[$3])\n01-05                          HashToRandomExchange(dist0=[[$0]], dist1=[[$1]], dist2=[[$2]])\n02-01                            UnorderedMuxExchange\n03-01                              Project(Sensor=[$0], Sensorname=[$1], Besitzer=[$2], sendung=[$3], E_X_P_R_H_A_S_H_F_I_E_L_D=[hash32AsDouble($2, hash32AsDouble($1, hash32AsDouble($0, 1301011:BIGINT)))])\n03-02                                HashAgg(group=[{0, 1, 

## Aufgabe 2 c

1. Die Datenmenge unter `dfs.bdapi.sensorslastday` ändert sich in Intervallen.  Finden Sie einen Weg, diese Datenmenge nutzbar zu machen. (Tipp: Drill bietet Funktionen für komplexe Datentypen an, z.B. [FLATTEN()](https://drill.apache.org/docs/flatten/)). Die Antwort Ihrer Query sollte die Felder `timewindow`, `celsius` und `humidity` liefern.
2. Geben Sie mit einer einzigen Query die beiden Datenpunkte der extremen Temperaturen (`celsius`) aus, d.h. die Zeile mit der höchsten Temeperatur und die Zeile mit der niedrigsten Temperatur.
3. Optional: Finden Sie für die Datenmenge `dfs.bdapi.sensorslastday` zu jedem Zeitraum heraus, ob es Tag ist oder Nacht. Das können Sie mithilfe der Datenmenge `dfs.weather.sunrise` tun. Nehmen Sie für den Sensor in der Datenmenge die Geokoordinaten `50.75410842895508`, `6.08587121963501` an.  

In [26]:
# Aufgabe 2c: 1.
csr.execute("""select query.flattenresultset.timewindow as timewindow, query.flattenresultset.celsius as celsius, query.flattenresultset.humidity as humidity
                        from (select flatten(resultset) as flattenresultset from bdapi.sensorslastday) query""")

results = csr.fetchall()

data = [(row.timewindow, row.celsius, row.humidity) for row in results]

table_headers = ["timewindow", "celsius", "humidity"]
print(tabulate(data, headers=table_headers, tablefmt="grid"))

+---------------------+-----------+------------+
| timewindow          |   celsius |   humidity |
+=====================+===========+============+
| 2023-05-25 01:45:00 |   20.457  |    38.3746 |
+---------------------+-----------+------------+
| 2023-05-25 01:45:00 |   23.2906 |    32.8892 |
+---------------------+-----------+------------+
| 2023-05-25 01:30:00 |   20.5135 |    38.4232 |
+---------------------+-----------+------------+
| 2023-05-25 01:30:00 |   23.3342 |    32.8159 |
+---------------------+-----------+------------+
| 2023-05-25 01:15:00 |   20.5992 |    38.3989 |
+---------------------+-----------+------------+
| 2023-05-25 01:15:00 |   23.3813 |    32.7645 |
+---------------------+-----------+------------+
| 2023-05-25 01:00:00 |   20.6259 |    37.8715 |
+---------------------+-----------+------------+
| 2023-05-25 01:00:00 |   23.4249 |    32.6618 |
+---------------------+-----------+------------+
| 2023-05-25 00:45:00 |   20.6678 |    38.3802 |
+-------------------

In [27]:
# Aufgabe 2c: 2.
csr.execute("""select MAX(CAST(query.flattenresultset.celsius as FLOAT)) as max_temp, MIN(CAST(query.flattenresultset.celsius as FLOAT)) as min_temp from 
(select flatten(resultset) as flattenresultset from bdapi.sensorslastday) query""")

results = csr.fetchall()

data = [(row.max_temp, row.min_temp) for row in results]

table_headers = ["Maximale Temperatur", "Minimale Temperatur"]
print(tabulate(data, headers=table_headers, tablefmt="grid"))

+-----------------------+-----------------------+
|   Maximale Temperatur |   Minimale Temperatur |
+=======================+=======================+
|               27.9635 |              -34.4797 |
+-----------------------+-----------------------+


In [13]:
# Aufabe 2c: 3. optional

_____